In [98]:
import sys
sys.path.append("../..")
import importlib
import clean_data

importlib.reload(clean_data)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import classification_report, log_loss, f1_score, balanced_accuracy_score

from clean_data import feature_engineering_days

In [99]:
df = pd.read_csv("../../data/austin/raw/aac_intakes.csv")
outcomes = pd.read_csv("../../data/austin/raw/aac_intakes_outcomes.csv")

df["datetime"] = pd.to_datetime(df["datetime"])
outcomes["intake_datetime"] = pd.to_datetime(outcomes["intake_datetime"])

df = df.merge(
    outcomes[["animal_id_intake", "intake_datetime", "time_in_shelter_days", "outcome_type"]],
    left_on=["animal_id", "datetime"],
    right_on=["animal_id_intake", "intake_datetime"],
    how="inner"
)

df = df[df["animal_type"] == "Dog"].copy()

In [100]:
pd.crosstab(
    df["intake_type"],
    df["outcome_type"]
)

outcome_type,Adoption,Died,Disposal,Euthanasia,Missing,Return to Owner,Rto-Adopt,Transfer
intake_type,,,,,,,,
Euthanasia Request,5,3,0,147,0,4,0,23
Owner Surrender,5050,29,2,475,2,551,29,2628
Public Assist,427,8,2,151,1,3026,12,462
Stray,14998,98,6,728,12,9610,98,6501


In [113]:
pd.crosstab(
    df["intake_condition"],
    df["outcome_type"]
)

outcome_type,Adoption,Transfer
intake_condition,,
Aged,55,48
Feral,2,6
Injured,576,448
Normal,19342,8312
Nursing,239,276
Other,29,29
Pregnant,14,14
Sick,218,458


In [101]:
df = df[df["outcome_type"].isin(["Adoption", "Transfer"])].copy()

In [102]:
pd.crosstab(
    df["outcome_type"],
    pd.cut(
        df["time_in_shelter_days"],
        bins=[-float("inf"), 4, 10, float("inf")],
        labels=["<=4", "4-10", ">10"]
    ),
    normalize="index"
).mul(100).round(2)

time_in_shelter_days,<=4,4-10,>10
outcome_type,,,
Adoption,12.47,47.10,40.43
Transfer,32.31,38.84,28.85


In [103]:
df = feature_engineering_days(df)

df_dogs = df[df["animal_type"] == "Dog"].copy()
df_dogs = df_dogs.sort_values("intake_datetime").reset_index(drop=True)

bins = [-np.inf, 4, 10, np.inf]
df_dogs["target"] = pd.cut(
    df_dogs["time_in_shelter_days"], bins=bins, labels=[0, 1, 2]
).astype(int)

In [104]:
drop_cols = [
    "target", "time_in_shelter_days", "time_in_shelter", "intake_datetime", 
    "animal_type", "young_and_cat", "datetime", "outcome_type"
]

X = df_dogs.drop(columns=drop_cols, errors="ignore")
y = df_dogs["target"]

#categorical features which will be available
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].fillna("Unknown").astype(str)

#70/15/15
n = len(X)
i1, i2 = int(n * 0.70), int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

print("Split:", len(X_train), len(X_val), len(X_test))

Split: 21046 4510 4510


In [105]:
model = CatBoostClassifier(
    iterations=1200, learning_rate=0.03, depth=6,
    loss_function="MultiClass", random_seed=42, verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

0:	learn: 1.0848582	test: 1.0845786	best: 1.0845786 (0)	total: 57.5ms	remaining: 1m 8s
100:	learn: 0.7818472	test: 0.8069318	best: 0.8069318 (100)	total: 6.89s	remaining: 1m 15s
200:	learn: 0.7500903	test: 0.7888565	best: 0.7888453 (199)	total: 15.2s	remaining: 1m 15s
300:	learn: 0.7340502	test: 0.7815056	best: 0.7814580 (299)	total: 22.8s	remaining: 1m 7s
400:	learn: 0.7202430	test: 0.7773988	best: 0.7773618 (381)	total: 29.4s	remaining: 58.6s
500:	learn: 0.7089527	test: 0.7742052	best: 0.7739991 (491)	total: 36.8s	remaining: 51.3s
600:	learn: 0.7001167	test: 0.7726827	best: 0.7726827 (600)	total: 44.8s	remaining: 44.6s
700:	learn: 0.6925680	test: 0.7719985	best: 0.7719789 (697)	total: 53.2s	remaining: 37.8s
800:	learn: 0.6856471	test: 0.7714409	best: 0.7714409 (800)	total: 1m 2s	remaining: 30.9s
900:	learn: 0.6790360	test: 0.7716925	best: 0.7712857 (825)	total: 1m 9s	remaining: 23s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7712857011
bestIteration = 825

Sh

CatBoostClassifier(depth=6, iterations=1200, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=100)

In [106]:
X_train.columns

Index(['breed', 'color', 'intake_condition', 'intake_type', 'name',
       'intake_hour', 'month_sin', 'month_cos', 'sex', 'intact',
       'is_mixed_breed', 'primary_breed', 'is_mixed_color', 'primary_color',
       'has_name', 'name_length', 'age_days', 'day_of_week', 'month', 'year',
       'is_weekend', 'is_puppy', 'is_senior', 'age_log', 'puppy_and_intact',
       'stray_and_intact', 'breed_age'],
      dtype='object')

# Evaluation


In [107]:
probs = model.predict_proba(X_test)
y_pred = model.classes_[np.argmax(probs, axis=1)]

#Log Loss
ll = log_loss(y_test, probs, labels=model.classes_)

#Multiclass Brier Score
y_onehot = (
    np.asarray(y_test)[:, None]
    == np.asarray(model.classes_)[None, :]
).astype(int)

brier = np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

#Macro F1
f1 = f1_score(y_test, y_pred, average="macro")

#Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Log Loss:          {ll:.4f}")
print(f"Brier Score:       {brier:.4f}")
print(f"Macro F1:          {f1:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Log Loss:          0.7529
Brier Score:       0.4494
Macro F1:          0.6529
Balanced Accuracy: 0.6469

Classification report:
              precision    recall  f1-score   support

           0       0.66      0.50      0.57       960
           1       0.71      0.73      0.72      2091
           2       0.63      0.71      0.67      1459

    accuracy                           0.67      4510
   macro avg       0.67      0.65      0.65      4510
weighted avg       0.67      0.67      0.67      4510



In [108]:
from sklearn.metrics import precision_recall_curve, average_precision_score

classes = list(model.classes_)
i2 = classes.index(2)

p2_val = model.predict_proba(X_val)[:, i2]
p2_test = model.predict_proba(X_test)[:, i2]

# Wybór progu na walidacji: najwyższa precyzja przy recall >= 0.6
prec, rec, thr = precision_recall_curve((y_val == 2).astype(int), p2_val)
ok = rec[:-1] >= 0.8
threshold = thr[ok][np.argmax(prec[:-1][ok])]
print(f"Próg dla klasy >10 dni: {threshold:.3f}")

# Ocena na teście
flag = p2_test >= threshold
y_long = (y_test == 2).astype(int)
tp = np.sum(flag & (y_long == 1))
print(f"Oznaczone: {flag.mean():.1%} zwierząt")
print(f"Precyzja: {tp / max(flag.sum(), 1):.3f}")
print(f"Recall:   {tp / y_long.sum():.3f}")
print(f"PR-AUC (klasa >10): {average_precision_score(y_long, p2_test):.3f}")

Próg dla klasy >10 dni: 0.323
Oznaczone: 45.3% zwierząt
Precyzja: 0.569
Recall:   0.797
PR-AUC (klasa >10): 0.695


In [109]:
def predict_dog(model, dog):
    probs = model.predict_proba(dog)[0]

    return pd.DataFrame({
        "Time in shelter": ["≤4 days", "4–10 days", ">30 days"],
        "Probability (%)": [
            round(probs[list(model.classes_).index(i)] * 100, 2)
            for i in [0, 1, 2]
        ]
    })


In [110]:
dog = X_test.iloc[[0]]
print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])
display(predict_dog(model, dog))

5.06875


,Time in shelter,Probability (%)
0,≤4 days,18.63
1,4–10 days,61.36
2,>30 days,20.01


In [111]:
dog = X_test.iloc[[30]]

print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])

display(predict_dog(model, dog))

1.88125


,Time in shelter,Probability (%)
0,≤4 days,44.97
1,4–10 days,18.96
2,>30 days,36.07


In [112]:
long_stay = X_test[df_dogs.loc[X_test.index, "time_in_shelter_days"] > 60]

dog = long_stay.iloc[[0]]

print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])
display(predict_dog(model, dog))


82.06527777777778


,Time in shelter,Probability (%)
0,≤4 days,0.61
1,4–10 days,11.79
2,>30 days,87.60
